In [2]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
import ast

In [45]:
path = "../../data/raw/movie_data_1"
movie_metadata = pd.read_csv(path + "/movies_metadata.csv")
movie_keywords = pd.read_csv(path + "/keywords.csv")
movie_credits = pd.read_csv(path + "/credits.csv")
movie_ratings = pd.read_csv(path + "/ratings.csv")

/tmp/ipykernel_28269/2486341617.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movie_metadata = pd.read_csv(path + "/movies_metadata.csv")


##Notes
Multiply ratings with 2 to make it uniform to the metadata
extract words from the keywords dictionary
extract important cast(first 3?) and director from the dataset
cols_to_use in metadata ["belongs_to_collection","genres","id","original_title","overview","popularity","poster_path","original_language","production_countries","production_companies","release_date","runtime","title","vote_average","vote_count"]
use belongs_to_collection for franchise flag maybe. 
extract genres as words,

In [46]:
#turn dtype of id columns to int
movie_metadata['id'] = pd.to_numeric(movie_metadata['id'], errors='coerce').astype('Int64')

In [47]:
movie_credits["cast"] = movie_credits["cast"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
movie_credits["crew"] = movie_credits["crew"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])

In [48]:
movie_credits["top_5_actors"] = movie_credits["cast"].apply(lambda x: [actor["name"] for actor in x[:5]])
movie_credits["director"] = movie_credits["crew"].apply(lambda x: [crew_member["name"] for crew_member in x if crew_member["job"] == "Director"])
movie_metadata = pd.merge(movie_metadata,movie_credits[["id", "top_5_actors", "director"]], on="id", how="left")

In [49]:

movie_keywords["keywords"] = movie_keywords["keywords"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
movie_keywords["keywords"] = movie_keywords["keywords"].apply(lambda x: [keyword["name"] for keyword in x])
movie_metadata = pd.merge(movie_metadata,movie_keywords[["id", "keywords"]], on="id", how="left")

In [50]:
movie_metadata.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,top_5_actors,director,keywords
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",[John Lasseter],"[jealousy, toy, boy, friendship, friends, riva..."
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...",[Joe Johnston],"[board game, disappearance, based on children'..."
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",[Howard Deutch],"[fishing, best friend, duringcreditsstinger, o..."
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0,"[Whitney Houston, Angela Bassett, Loretta Devi...",[Forest Whitaker],"[based on novel, interracial relationship, sin..."
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0,"[Steve Martin, Diane Keaton, Martin Short, Kim...",[Charles Shyer],"[baby, midlife crisis, confidence, aging, daug..."


In [51]:
#parse genres,belongs_to_collection,production_companies
movie_metadata["genres"] = movie_metadata["genres"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
movie_metadata["genres"] = movie_metadata["genres"].apply(lambda x: [genre["name"] for genre in x])
movie_metadata["belongs_to_collection"] = movie_metadata["belongs_to_collection"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else {})
movie_metadata["collection_name"] = movie_metadata["belongs_to_collection"].apply(lambda x: x["name"] if isinstance(x, dict) and "name" in x else None)
movie_metadata["production_companies"] = movie_metadata["production_companies"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
movie_metadata["production_companies"] = movie_metadata["production_companies"].apply(lambda x: [company["name"] for company in x] if isinstance(x, list) else [])

In [52]:
cols_to_keep = ["title","genres","id","original_title","overview","popularity","poster_path","original_language","production_companies","release_date","runtime","vote_average","vote_count","collection_name","keywords","top_5_actors","director"]
movie_metadata_final = movie_metadata[cols_to_keep]

In [65]:
#drop rows with missing overview,vote average, keywords,
movie_metadata_final = movie_metadata_final.dropna(subset=["overview","vote_average","keywords"])
#drop genres with empty list
movie_metadata_final = movie_metadata_final[movie_metadata_final["genres"].apply(lambda x: len(x) > 0)]
#drop rows with empty list for keywords
movie_metadata_final = movie_metadata_final[movie_metadata_final["keywords"].apply(lambda x: len(x) > 0)]

In [63]:
#show number of rows with empty lists for the list columns
list_columns = ["genres","production_companies","keywords","top_5_actors","director"]
for col in list_columns:
    empty_list_rows = movie_metadata_final[movie_metadata_final[col].apply(lambda x: isinstance(x, list) and len(x) == 0)]
    print(f"Rows with empty lists in column '{col}': {len(empty_list_rows)}")


Rows with empty lists in column 'genres': 0
Rows with empty lists in column 'production_companies': 9744
Rows with empty lists in column 'keywords': 12205
Rows with empty lists in column 'top_5_actors': 1713
Rows with empty lists in column 'director': 539


In [71]:
movie_metadata_final[movie_metadata_final["id"] == 27205]

,title,genres,id,original_title,overview,popularity,poster_path,original_language,production_companies,release_date,runtime,vote_average,vote_count,collection_name,keywords,top_5_actors,director
15576,Inception,"[Action, Thriller, Science Fiction, Mystery, A...",27205,Inception,"Cobb, a skilled thief who commits corporate es...",29.108149,/qmDpIHrmpJINaRKAfWQfftjCdyi.jpg,en,"[Legendary Pictures, Warner Bros., Syncopy]",2010-07-14,148.0,8.1,14075.0,None,"[loss of lover, dream, kidnapping, sleep, subc...","[Leonardo DiCaprio, Joseph Gordon-Levitt, Elle...",[Christopher Nolan]


In [69]:
full_movie_data = pd.read_csv("../../data/raw/TMDB_movie_dataset_v11.csv")

In [79]:
full_movie_data

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,budget,homepage,imdb_id,original_language,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,160000000,https://www.warnerbros.com/movies/inception,tt1375666,en,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,165000000,http://www.interstellarmovie.net/,tt0816692,en,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,185000000,https://www.warnerbros.com/movies/dark-knight/,tt0468569,en,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,237000000,https://www.avatar.com/movies/avatar,tt0499549,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,220000000,https://www.marvel.com/movies/the-avengers,tt0848228,en,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1382078,852681,London and the Muse,0.000,0,Released,2015-12-12,0,19,False,NaN,0,NaN,tt5004252,en,London and the Muse,A former child prodigy painter must face her m...,0.600,NaN,NaN,"Mystery, Thriller",NaN,United States of America,NaN,NaN
1382079,852683,Night of the Pigs,0.000,0,Released,2021-10-02,0,13,False,/cQtPQSDPSRsyviUalTTG32ssENm.jpg,1500,NaN,NaN,sv,Grisarnas Natt,A gang of metalheads is partying in the woods ...,0.600,/kKN5v4jBGUGK2EB5OIR1mX9uDak.jpg,NaN,Horror,NaN,Sweden,Swedish,"murder, drugs, psychosis"
1382080,852685,Auf den Spuren von GutsMuths,0.000,0,Released,1989-03-03,0,26,False,NaN,0,NaN,tt19730642,de,Auf den Spuren von GutsMuths,NaN,0.600,/9mC6P36JijiDLJzHMH3LajVVKIi.jpg,NaN,Documentary,DEFA Studio für Dokumentarfilme,"East Germany, Germany",German,NaN
1382081,852686,Fragments of a Lost Palestine,0.000,0,Released,2010-01-01,0,0,False,NaN,0,NaN,NaN,it,Fragments of a Lost Palestine,NaN,0.600,/qcGarhUy5fThgYc2SQidhNbxiaB.jpg,NaN,Documentary,NaN,France,Arabic,NaN


In [72]:
#replace post_path on movie_metadata_final with the one in full_movie_data based on id column
full_movie_data_subset = full_movie_data[["id", "poster_path"]]
movie_metadata_final = pd.merge(movie_metadata_final.drop(columns=["poster_path"]), full_movie_data_subset, on="id", how="left")

In [75]:
#if there is a collection name create a flag for has_franchise
movie_metadata_final["has_franchise"] = movie_metadata_final["collection_name"].apply(lambda x: 1 if pd.notnull(x) else 0)

In [76]:
movie_metadata_final.to_csv("../../data/processed/movie_metadata_final.csv", index=False)

In [78]:
movie_metadata_final.columns

Index(['title', 'genres', 'id', 'original_title', 'overview', 'popularity',
       'original_language', 'production_companies', 'release_date', 'runtime',
       'vote_average', 'vote_count', 'collection_name', 'keywords',
       'top_5_actors', 'director', 'poster_path', 'has_franchise'],
      dtype='object')

In [80]:
movie_metadata_final[movie_metadata_final["title"] == "The Immortals"]

,title,genres,id,original_title,overview,popularity,original_language,production_companies,release_date,runtime,vote_average,vote_count,collection_name,keywords,top_5_actors,director,poster_path,has_franchise
17029,The Immortals,"[Action, Crime, Drama, Thriller]",29231,The Immortals,An elaborate heist unites 8 strangers in a sim...,0.825042,en,"[Nu Image Films, Mondofin B.V., End Productions]",1995-10-05,98.0,7.8,5.0,None,"[nightclub, heist, night club, henchman, evil ...","[Eric Roberts, Joe Pantoliano, Tia Carrere, To...",[Brian Grant],/rShuJsXmzT4pzUm6aBO5NIpq1uw.jpg,0


In [ ]:
#show unique genres in genres column
import pandas as pd




{'A', 'l', 'V', "'", 'c', 'W', 'S', ' ', 'v', 'i', 'T', 'D', '[', 'a', 't', 'y', ',', 'R', 'h', 'M', 'o', 'm', 'd', 'r', ']', 'C', 'u', 's', 'g', 'e', 'F', 'H', 'n'}


In [9]:
#parse genres column currently its string representation of list, convert it to list
import ast
movie_metadata_final["genres"] = movie_metadata_final["genres"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])

In [10]:
unique_genres = set()
for genres_list in movie_metadata_final["genres"]:
    unique_genres.update(genres_list)
print(unique_genres)

{'Foreign', 'Crime', 'Animation', 'War', 'TV Movie', 'Romance', 'Adventure', 'Fantasy', 'Mystery', 'Family', 'Documentary', 'Music', 'Horror', 'History', 'Thriller', 'Comedy', 'Action', 'Western', 'Science Fiction', 'Drama'}


## User Data

In [2]:
import pandas as pd
movie_metadata_final = pd.read_csv("../../data/processed/movie_metadata_final.csv")

# Load links.csv for MovieLens movieId → TMDB tmdbId mapping
links = pd.read_csv("../../data/raw/movie_data_1/links.csv")
links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce")
links = links.dropna(subset=["tmdbId"])
links["tmdbId"] = links["tmdbId"].astype("int64")
links["movieId"] = links["movieId"].astype("int64")

# Build movieId → tmdbId mapping
ml_to_tmdb = dict(zip(links["movieId"], links["tmdbId"]))
catalog_tmdb_ids = set(movie_metadata_final["id"].astype("int64").tolist())

print(f"Links entries: {len(ml_to_tmdb):,}")
print(f"Catalog TMDB IDs: {len(catalog_tmdb_ids):,}")

Links entries: 45,624
Catalog TMDB IDs: 30,543


In [4]:
import pandas as pd
user_df = pd.read_csv("../../data/raw/movie_data_1/ratings.csv")

In [2]:
len(user_df)

26024289

In [5]:
# Map MovieLens movieId → TMDB tmdbId using links.csv
user_df["tmdbId"] = user_df["movieId"].map(ml_to_tmdb)
print(f"Rows before mapping: {len(user_df):,}")
user_df = user_df.dropna(subset=["tmdbId"])
user_df["tmdbId"] = user_df["tmdbId"].astype("int64")
print(f"Rows after mapping to tmdbId: {len(user_df):,}")

# Filter to movies that exist in our catalog
user_df = user_df[user_df["tmdbId"].isin(catalog_tmdb_ids)]
print(f"Rows after catalog filter: {len(user_df):,}")

# Replace movieId with tmdbId for downstream consistency
user_df = user_df.drop(columns=["movieId"]).rename(columns={"tmdbId": "movieId"})

# Convert timestamp to datetime
from datetime import datetime
user_df["timestamp"] = pd.to_datetime(user_df["timestamp"], unit="s")
print(f"Date range: {user_df['timestamp'].min()} — {user_df['timestamp'].max()}")
print(f"Unique movies: {user_df['movieId'].nunique():,}")
print(f"Unique users: {user_df['userId'].nunique():,}")

Rows before mapping: 26,024,289
Rows after mapping to tmdbId: 26,010,786
Rows after catalog filter: 25,555,768
Date range: 1995-01-09 11:46:44 — 2017-08-04 06:57:50
Unique movies: 30,023
Unique users: 270,813


In [6]:
len(user_df)

25555768

In [7]:
user_df.to_csv("../../data/processed/user_ratings.csv", index=False)